In [1]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import optuna

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
torch.manual_seed(42)

In [4]:
df_train = pd.read_csv('dataset/fashion-mnist_train.csv')
df_test = pd.read_csv('dataset/fashion-mnist_test.csv')

In [5]:
df_train.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
X_train, y_train = df_train.iloc[:,1:], df_train.iloc[:,0]
X_test, y_test = df_test.iloc[:,1:], df_test.iloc[:,0]

In [7]:
X_train = X_train/255
X_test = X_test/255

In [8]:
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features.values, dtype=torch.float32)
        self.labels = torch.tensor(labels.values, dtype=torch.long)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [9]:
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

In [10]:
class MyNN(nn.Module):
    def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate):
        super().__init__()
        layers = []
        for i in range(num_hidden_layers):
            layers.append(nn.Linear(input_dim, neurons_per_layer))
            layers.append(nn.BatchNorm1d(neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            input_dim = neurons_per_layer
        layers.append(nn.Linear(neurons_per_layer, output_dim))
        self.model = nn.Sequential(*layers)

    def forward(self, X):
        return self.model(X)

In [11]:
def objective(trial):
    num_hidden_layers = trial.suggest_int("num_hidden_layers",1,5)
    neurons_per_layer = trial.suggest_int("neurons_per_layer",8,128,step=8)
    epochs = trial.suggest_int("epochs",10,100,step=10)
    learning_rate = trial.suggest_float("learning_rate",1e-5,1e-1,log=True)
    dropout_rate = trial.suggest_float("dropout_rate",0.1,0.5,step=0.1)
    batch_size = trial.suggest_categorical("batch_size",[16,32,64,128])
    optimizer_name=trial.suggest_categorical("optimizer_name",["Adam","SGD","RMSprop"])
    weight_decay = trial.suggest_float("weight_decay",1e-5,1e-3,log=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

    input_dim = 784
    output_dim = 10

    model = MyNN(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate)
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    if optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(),lr=learning_rate,weight_decay=weight_decay)
    elif optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(),lr=learning_rate,weight_decay=weight_decay)
    else:
        optimizer = optim.RMSprop(model.parameters(),lr=learning_rate,weight_decay=weight_decay)

    for epoch in range(epochs):
        for batch_features, batch_labels in train_loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)
            outputs = model(batch_features)
            loss = criterion(outputs, batch_labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for batch_features, batch_labels in test_loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            outputs = model(batch_features)
            _,predicted = torch.max(outputs, 1)
            total += batch_labels.shape[0]
            correct += (predicted == batch_labels).sum().item()
        accuracy = correct/total
    print(f"Accuracy: {accuracy}")
    return accuracy

In [12]:
study = optuna.create_study(direction = 'maximize')

[I 2026-06-15 00:38:31,931] A new study created in memory with name: no-name-ef7c1c2d-30e5-4e3d-a59c-e742a55c110a


In [13]:
study.optimize(objective, n_trials=10)

[I 2026-06-15 00:40:26,040] Trial 0 finished with value: 0.8596 and parameters: {'num_hidden_layers': 1, 'neurons_per_layer': 24, 'epochs': 30, 'learning_rate': 0.0011725880161975028, 'dropout_rate': 0.2, 'batch_size': 32, 'optimizer_name': 'SGD', 'weight_decay': 3.173518038003456e-05}. Best is trial 0 with value: 0.8596.


Accuracy: 0.8596


[I 2026-06-15 00:41:09,770] Trial 1 finished with value: 0.8478 and parameters: {'num_hidden_layers': 1, 'neurons_per_layer': 24, 'epochs': 10, 'learning_rate': 0.0018095984191679965, 'dropout_rate': 0.30000000000000004, 'batch_size': 32, 'optimizer_name': 'Adam', 'weight_decay': 0.0005547109412159873}. Best is trial 0 with value: 0.8596.


Accuracy: 0.8478


[I 2026-06-15 00:45:46,812] Trial 2 finished with value: 0.8843 and parameters: {'num_hidden_layers': 2, 'neurons_per_layer': 112, 'epochs': 90, 'learning_rate': 3.042039191327065e-05, 'dropout_rate': 0.5, 'batch_size': 64, 'optimizer_name': 'RMSprop', 'weight_decay': 0.0002679699272743275}. Best is trial 2 with value: 0.8843.


...

Accuracy: 0.8865


[I 2026-06-15 01:24:53,080] Trial 8 finished with value: 0.3864 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 104, 'epochs': 70, 'learning_rate': 0.06906586563440867, 'dropout_rate': 0.30000000000000004, 'batch_size': 16, 'optimizer_name': 'Adam', 'weight_decay': 0.0002137697964091759}. Best is trial 7 with value: 0.8865.


Accuracy: 0.3864


[I 2026-06-15 01:25:40,861] Trial 9 finished with value: 0.8062 and parameters: {'num_hidden_layers': 4, 'neurons_per_layer': 48, 'epochs': 20, 'learning_rate': 0.020844079640537447, 'dropout_rate': 0.5, 'batch_size': 128, 'optimizer_name': 'SGD', 'weight_decay': 0.0002488129157150387}. Best is trial 7 with value: 0.8865.


Accuracy: 0.8062


In [14]:
study.best_value

0.8865

In [15]:
study.best_params

{'num_hidden_layers': 4,
 'neurons_per_layer': 96,
 'epochs': 10,
 'learning_rate': 8.953306109917998e-05,
 'dropout_rate': 0.1,
 'batch_size': 128,
 'optimizer_name': 'Adam',
 'weight_decay': 4.1845353807104765e-05}